# Single-Qubit Gates

Explore X, Y, Z, H, S, T, and rotation gates. Visualize state transformations.

**Objectives:**
- Apply each standard single-qubit gate and observe its effect
- Understand the relationship between gate parameters and measurement probabilities
- Visualize qubit states using the Bloch circle representation

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->


In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

from lib.utils.visualization import plot_bloch_angles

device = LocalSimulator()

def measure_probabilities(circuit, shots=2000):
    """Run a circuit and return P(|0>) and P(|1>)."""
    result = device.run(circuit, shots=shots).result()
    counts = result.measurement_counts
    total = sum(counts.values())
    p0 = counts.get('0', 0) / total
    p1 = counts.get('1', 0) / total
    return p0, p1

## 1. Pauli Gates (X, Y, Z)

The Pauli gates are 180-degree rotations about the X, Y, and Z axes of the Bloch sphere.

In [ ]:
# X gate (NOT gate): flips |0> to |1> and vice versa
circuit_x = Circuit().x(0)
p0, p1 = measure_probabilities(circuit_x)
print(f"X|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}")
print("Expected: P(|0>) = 0.000, P(|1>) = 1.000")

# Z gate: phase flip (no effect on |0>, adds minus sign to |1>)
# Z|0> = |0>, so measurements are unchanged
circuit_z = Circuit().z(0)
p0, p1 = measure_probabilities(circuit_z)
print(f"\nZ|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}")
print("Z has no visible effect on |0> in the computational basis.")

# But Z *does* affect superposition states:
# H puts us in |+>, then Z flips to |->, then H maps back to |1>
circuit_hzh = Circuit().h(0).z(0).h(0)
p0, p1 = measure_probabilities(circuit_hzh)
print(f"\nHZH|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}")
print("HZH acts like X -- the Z flipped the relative phase in superposition.")

## 2. Hadamard Gate (H)

The Hadamard gate creates superposition from basis states and is self-inverse (H*H = I).

In [ ]:
# H|0> = |+> = (|0> + |1>) / sqrt(2)
circuit_h = Circuit().h(0)
p0, p1 = measure_probabilities(circuit_h)
print(f"H|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}  (expect ~0.5 each)")

# H is self-inverse: HH|0> = |0>
circuit_hh = Circuit().h(0).h(0)
p0, p1 = measure_probabilities(circuit_hh)
print(f"HH|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}  (back to |0>)")

# H|1> = |-> = (|0> - |1>) / sqrt(2)
# Still 50/50 in computational basis -- phase isn't visible in Z-measurement
circuit_xh = Circuit().x(0).h(0)
p0, p1 = measure_probabilities(circuit_xh)
print(f"H|1>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}  (also ~0.5, phase hidden)")

## 3. Rotation Gates (Rx, Ry, Rz)

Rotation gates take a continuous angle parameter, enabling fine-grained state preparation.
Ry(theta) is particularly useful: it rotates from |0> toward |1> without introducing complex phases.

In [ ]:
# Sweep Ry(theta) from 0 to pi and plot P(|1>) vs theta
thetas = np.linspace(0, np.pi, 20)
p1_values = []

for theta in thetas:
    circuit = Circuit().ry(0, theta)
    _, p1 = measure_probabilities(circuit, shots=2000)
    p1_values.append(p1)

# Theoretical: P(|1>) = sin^2(theta/2)
theoretical = np.sin(thetas / 2) ** 2

plt.figure(figsize=(8, 4))
plt.plot(thetas, p1_values, 'o', label='Measured', markersize=5)
plt.plot(thetas, theoretical, '-', label='Theory: sin^2(theta/2)', color='orange')
plt.xlabel('theta (radians)')
plt.ylabel('P(|1>)')
plt.title('Ry(theta)|0>: Probability of measuring |1>')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize specific states on the Bloch circle
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

states = [
    (0, 0, "|0> (theta=0)"),
    (np.pi/4, 0, "Ry(pi/4)|0>"),
    (np.pi/2, 0, "|+> (theta=pi/2)"),
    (np.pi, 0, "|1> (theta=pi)"),
]

for ax, (theta, phi, title) in zip(axes, states):
    fig_single = plot_bloch_angles(theta, phi, title=title)
    plt.close(fig_single)

# Redraw inline for side-by-side comparison
for i, (theta, phi, title) in enumerate(states):
    ax = axes[i]
    circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=1.5)
    ax.add_patch(circle)
    x = np.sin(theta) * np.cos(phi)
    z = np.cos(theta)
    ax.arrow(0, 0, x * 0.85, z * 0.85, head_width=0.06, head_length=0.04, fc='#ff9900', ec='#ff9900')
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title(title, fontsize=10)
    ax.text(0, 1.15, '|0>', ha='center', fontsize=9)
    ax.text(0, -1.15, '|1>', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Phase Gates (S and T)

S and T gates add phase to the |1> component. Like Z, they have no effect on |0> in the computational basis,
but they affect interference when combined with Hadamard gates.

In [ ]:
# S gate: pi/2 phase on |1>. S*S = Z.
# Demonstrate: H -> S -> H maps |0> to a state with P(|1>) = 0.5
circuit_hsh = Circuit().h(0).s(0).h(0)
p0, p1 = measure_probabilities(circuit_hsh)
print(f"HSH|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}")

# T gate: pi/4 phase on |1>. T*T = S.
circuit_hth = Circuit().h(0).t(0).h(0)
p0, p1 = measure_probabilities(circuit_hth)
print(f"HTH|0>: P(|0>) = {p0:.3f}, P(|1>) = {p1:.3f}")
print(f"  Theory: P(|1>) = sin^2(pi/8) = {np.sin(np.pi/8)**2:.3f}")

# Verify T*T = S by comparing H-TT-H vs H-S-H
circuit_htth = Circuit().h(0).t(0).t(0).h(0)
p0_tt, p1_tt = measure_probabilities(circuit_htth)
print(f"\nHTTH|0>: P(|1>) = {p1_tt:.3f} (should match HSH = {p1:.3f})")

## 5. Exercises

Three exercises to make the gates stick. Each has tiered hints -- expand only
what you need -- and a check cell that tells you when you have it.

### Exercise 1 — Build and unmask the |-> state

Create the `|->` state using H then Z (or, equivalently, X then H) and
confirm it measures 50/50. Then show that one more H maps `|->` to `|1>` --
interference makes the hidden phase visible.

Define `minus_probs`: the `(P0, P1)` pair from measuring your `|->` circuit,
and `back_probs`: the `(P0, P1)` pair from the same gate sequence with one
more H appended. Use `measure_probabilities(...)` for both.

<details><summary>Hint 1 — nudge</summary>

`|+>` and `|->` are indistinguishable in the computational basis -- both
measure 50/50. Section 1's HZH sandwich showed how a final H turns a hidden
phase into a visible outcome. Which state would that H send back to `|0>`,
and which to `|1>`?

</details>
<details><summary>Hint 2 — approach</summary>

Build one circuit for `|->` (two gates) and measure it with
`measure_probabilities(...)`. Then build a second circuit that appends
`h(0)` to the same gate sequence and measure again -- if the phase flip is
really there, all the weight lands on `|1>`.

</details>

In [ ]:
# Exercise 1: Build the |-> state, then unmask its phase with a second H.
# Define: minus_probs -- (P0, P1) for the |-> circuit, and back_probs --
# (P0, P1) for the same sequence with one more H appended.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(minus_probs) == 2, "minus_probs should be the (P0, P1) pair"
    assert abs(sum(minus_probs) - 1.0) < 1e-6, "the two probabilities should sum to 1"
    assert abs(minus_probs[0] - 0.5) < 0.1, (
        "|-> should measure 50/50 in the computational basis"
    )
    assert len(back_probs) == 2, "back_probs should be the (P0, P1) pair"
    assert back_probs[1] > 0.98, (
        "with the extra H every shot should land on |1> -- if weight stays on "
        "|0>, the phase flip is missing"
    )

### Exercise 2 — Dial in a 75/25 split

Rotation gates prepare any bias you like. Find the Ry angle that gives
exactly 75% probability of measuring `|0>`, then verify it on the simulator.

Define `theta_75`: that angle in radians, and `p0_75`: the measured P(|0>)
from running `Circuit().ry(0, theta_75)` with at least 2000 shots.

<details><summary>Hint 1 — nudge</summary>

Section 3 plotted the theory curve for Ry: the measurement probabilities are
smooth trig functions of theta. Finding the angle is inverting that curve at
0.75 -- no searching required.

</details>
<details><summary>Hint 2 — approach</summary>

Write P(|0>) for `Ry(theta)|0>` as a squared cosine of half the angle, set
it equal to 0.75, and solve for theta on paper (or with `np.arccos` and
`np.sqrt`). Then build the one-gate circuit and confirm the measured P(|0>)
with `measure_probabilities(..., shots=...)`.

</details>

In [ ]:
# Exercise 2: Find the Ry angle that gives exactly 75% probability of |0>.
# Define: theta_75 -- the angle in radians, and p0_75 -- the measured P(|0>)
# from at least 2000 shots.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert 0 < abs(float(theta_75)) < 2 * np.pi, (
        "theta_75 should be a nonzero rotation angle in radians"
    )
    assert abs(np.cos(theta_75 / 2) ** 2 - 0.75) < 1e-6, (
        "Ry(theta_75)|0> should put exactly 75% of its weight on |0> -- "
        "revisit the inversion"
    )
    assert abs(p0_75 - 0.75) < 0.06, (
        "the measured P(|0>) should land within a few percent of 0.75"
    )

### Exercise 3 — Rx(pi) equals X, up to global phase

The Pauli gates are 180-degree rotations: `Rx(pi)` and `X` differ only by a
global phase, which no measurement can see. Verify that by comparing their
statistics.

Define `rx_pi_probs` and `x_probs`: the `(P0, P1)` pairs from measuring
`Circuit().rx(0, np.pi)` and `Circuit().x(0)` respectively.

<details><summary>Hint 1 — nudge</summary>

A global phase multiplies the entire state by one complex number of unit
size, so every |amplitude|^2 -- and therefore every measurement probability
-- is untouched. What should "the same gate up to global phase" mean for the
two sets of statistics?

</details>
<details><summary>Hint 2 — approach</summary>

Build the two one-gate circuits, run each through
`measure_probabilities(...)`, and compare the pairs -- both should put
essentially all the weight on the same basis state.

</details>

In [ ]:
# Exercise 3: Verify that Rx(pi) = X up to global phase by comparing measurements.
# Define: rx_pi_probs and x_probs -- (P0, P1) pairs for the two circuits.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert len(rx_pi_probs) == 2 and len(x_probs) == 2, (
        "record each result as a (P0, P1) pair"
    )
    assert x_probs[1] > 0.98, "X|0> should measure |1> on every shot"
    assert rx_pi_probs[1] > 0.98, (
        "Rx(pi)|0> should also measure |1> on every shot -- check the angle"
    )
    assert abs(rx_pi_probs[1] - x_probs[1]) < 0.05, (
        "the two circuits should give statistically identical results"
    )

## Summary

- **X gate**: bit flip (NOT), rotates |0> to |1>
- **Z gate**: phase flip, invisible on |0>/|1> but affects superpositions
- **H gate**: creates/destroys superposition, self-inverse
- **Ry(theta)**: smooth rotation from |0> toward |1>, P(|1>) = sin^2(theta/2)
- **S, T gates**: add phase to |1> component, affect interference patterns
- Phase effects are only visible through interference (e.g., H-gate-H sandwich)

**Next:** [03-multi-qubit-gates.ipynb](03-multi-qubit-gates.ipynb) -- create entanglement with CNOT, SWAP, and Toffoli gates.